# Project 3 — Part 1: Backbone Exploration (ResNet18)

## Goals
- Load pretrained ResNet18 (ImageNet)
- Understand its architecture (stages + residual blocks)
- Run inference on several images and present Top‑5 predictions
- Discuss strengths/limitations observed in inference

In [ ]:
import torch
from torchvision import models
from torchvision.utils import make_grid
import torchvision.transforms as T
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
device


## Environment setup
We import PyTorch + Torchvision, define a random seed for reproducibility, and choose a compute device:
- `cuda` if a supported NVIDIA GPU is available
- otherwise `cpu`

## What is a “backbone”?
A backbone is a feature extractor pretrained for classification (e.g., ImageNet).
For detection, we typically reuse the backbone’s convolutional feature maps and replace/extend the final head.

## Why ResNet?
ResNet introduces skip connections (residual learning) that make deeper networks easier to optimize.

## Loading a pretrained ResNet18 backbone (ImageNet)
Here we:
- Select pretrained weights (`ResNet18_Weights.DEFAULT`)
- Create the model and move it to `device`
- Switch to inference mode (`model.eval()`)
- Get the exact preprocessing pipeline required by these weights (`weights.transforms()`)
- Load the 1000 ImageNet class names (`weights.meta["categories"]`)

In [ ]:
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights).to(device)
model.eval()

# correct preprocessing for ImageNet weights
preprocess = weights.transforms()

# ImageNet class names in correct order (1000)
categories = weights.meta["categories"]

type(model), len(categories)

## Inspecting the architecture (modules)
Printing `model` shows the exact PyTorch module structure:
- Stem: `conv1` → `bn1` → `relu` → `maxpool`
- Stages: `layer1..layer4` (each is a sequence of residual BasicBlocks)
- Head: `avgpool` → `fc` (ImageNet classifier)

We’ll later reuse the backbone (up to `layer4`) for detection.

In [ ]:
print(model)

## Tracing the feature map shapes through the network
We attach forward hooks to major stages (conv1, layer1..layer4, etc.) and run a dummy input.
This shows how ResNet reduces spatial resolution (H×W) while increasing the number of channels (C).

In [ ]:
from collections import OrderedDict

def get_activation_shapes(m, x):
    shapes = OrderedDict()
    hooks = []

    def register(name, module):
        def hook(module, inp, out):
            shapes[name] = tuple(out.shape)
        hooks.append(module.register_forward_hook(hook))

    for name in ["conv1", "maxpool", "layer1", "layer2", "layer3", "layer4", "avgpool", "fc"]:
        register(name, getattr(m, name))

    with torch.inference_mode():
        _ = m(x)

    for h in hooks:
        h.remove()
    return shapes

dummy = torch.randn(1, 3, 224, 224, device=device)
get_activation_shapes(model, dummy)

## Helper functions for inference
- `predict_topk`: preprocesses a PIL image, runs the model, converts logits → probabilities (softmax), and returns the Top‑k predictions.
- `show`: displays the image cleanly inside the notebook.

We keep inference code in functions so the demo loop stays short and readable.

In [ ]:
@torch.inference_mode()
def predict_topk(pil_img, k=5):
    x = preprocess(pil_img).unsqueeze(0).to(device)  # [1,3,224,224]
    logits = model(x)[0]                              # [1000]
    probs = logits.softmax(dim=0)
    top = torch.topk(probs, k)
    return [(categories[i], float(p)) for p, i in zip(top.values, top.indices)]

def show_with_topk(img, preds, title=None):
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    ax[0].imshow(img)
    ax[0].axis("off")
    ax[0].set_title(title or "")

    text = "\n".join([f"{i+1}) {cls} — {pr:.3f}" for i, (cls, pr) in enumerate(preds)])
    ax[1].axis("off")
    ax[1].text(0.0, 1.0, text, va="top", family="monospace", fontsize=11)
    plt.show()
    
def show(img, title=None):
    plt.figure(figsize=(4,4))
    plt.imshow(img)
    plt.axis("off")
    if title:
        plt.title(title)
    plt.show()


In [ ]:
img_dir = Path("../data/backbone_demo")
paths = sorted(list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png")))

print("found", len(paths), "images")
paths[:5]

In [ ]:
for p in paths[:30]:
    img = Image.open(p).convert("RGB")
    preds = predict_topk(img, k=5)
    show_with_topk(img, preds, title=p.name)

## Inference demo (local images)
In this section we will:
- Load images from `data/backbone_demo/`
- Run the pretrained ResNet18 on each image
- Display the image and the Top‑5 ImageNet predictions (class + probability)

This fulfills the “inference demonstration” requirement for Part 1.